# Day 2 Session 3 -- Demos: LangGraph Orchestration & Workflow Design

This notebook contains the **instructor-led demos** for Session 3. Run each cell, observe the output, and follow along as the instructor explains each LangGraph pattern.

**Engineering context:** You are an engineer building workflow orchestration systems. Your users (consultants) need reliable, stateful pipelines that route work intelligently, refine outputs iteratively, and include human checkpoints. LangGraph lets you model these requirements as directed graphs with typed state, conditional edges, and cycles.

## Session Goal

By the end of this session, students will learn LangGraph's **5 core patterns**:

1. **Linear pipelines** with typed state -- sequential processing with a shared state object
2. **Conditional routing** -- branching logic that directs flow based on LLM decisions
3. **ReAct think-act-observe loops** -- cyclic agents that reason and use tools iteratively
4. **Cyclic refinement** -- iterative improvement until a quality threshold is met
5. **Human-in-the-loop gates** -- pause points requiring explicit human approval

These patterns model how real consulting engagement workflows operate: intake pipelines classify and route work, research agents gather intelligence iteratively, deliverables go through review cycles, and nothing reaches the client without partner sign-off.

In [ ]:
!pip install -q langchain langchain-openai langchain-core langgraph python-dotenv

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
import operator
import json
import os

print("All imports successful!")

---
## Demo 1: Linear Pipeline -- Modeling a Consulting Engagement Pipeline

A StateGraph has three core components:
- **State**: A TypedDict that holds all data flowing through the graph (the engagement tracker)
- **Nodes**: Python functions that read and update state (each node is a step in the pipeline)
- **Edges**: Connections that define the execution order (the workflow sequence)

Here we model a simple engagement intake pipeline: a client request arrives, we extract the scope, classify the industry sector, and format a preliminary engagement brief.

### Step (a): Define the State Schema

The state schema is the single most important design decision in a LangGraph workflow. It defines what data flows through the graph and what each node can read or write.

In [ ]:
# Demo 1a - Define the state schema (the engagement tracker)

class EngagementState(TypedDict):
    client_request: str       # Raw input from the client
    scope_summary: str        # Extracted and normalized scope
    industry_sector: str      # Classified industry vertical
    engagement_brief: str     # Final formatted brief

print("State schema defined: EngagementState")
print("Fields:", list(EngagementState.__annotations__.keys()))

**Key Concept:** The `TypedDict` state is the "engagement tracker" -- every node reads from it and writes back to it. Think of it as the shared context that flows through the pipeline. Unlike passing arguments between functions, every node has access to the full state but only updates the keys it owns.

### Step (b): Define Node Functions

Each node is a Python function that:
1. Takes the current state as input
2. Performs some processing
3. Returns a **partial** dictionary with only the keys it updates

In [ ]:
# Demo 1b - Define node functions (each takes state, returns partial state update)

def extract_scope_node(state: EngagementState) -> dict:
    """Extract and normalize the engagement scope from the client request."""
    return {"scope_summary": state["client_request"].strip().upper()}

def classify_industry_node(state: EngagementState) -> dict:
    """Classify the industry sector based on keywords in the request."""
    text = state["client_request"].lower()
    if any(kw in text for kw in ["bank", "financial", "fintech", "insurance"]):
        sector = "Financial Services"
    elif any(kw in text for kw in ["pharma", "health", "biotech", "hospital"]):
        sector = "Healthcare & Life Sciences"
    elif any(kw in text for kw in ["retail", "consumer", "e-commerce", "cpg"]):
        sector = "Consumer & Retail"
    else:
        sector = "Cross-Industry"
    return {"industry_sector": sector}

def format_brief_node(state: EngagementState) -> dict:
    """Format the preliminary engagement brief."""
    return {"engagement_brief": f"ENGAGEMENT BRIEF\nScope: {state['scope_summary']}\nSector: {state['industry_sector']}"}

print("Three node functions defined: extract_scope_node, classify_industry_node, format_brief_node")

### Step (c): Build the Graph and Run

Now we wire the nodes together into a directed graph with sequential edges.

In [ ]:
# Demo 1c - Build the graph and run

graph = StateGraph(EngagementState)

# Add nodes
graph.add_node("extract_scope", extract_scope_node)
graph.add_node("classify_industry", classify_industry_node)
graph.add_node("format_brief", format_brief_node)

# Define edges (the execution order)
graph.set_entry_point("extract_scope")
graph.add_edge("extract_scope", "classify_industry")
graph.add_edge("classify_industry", "format_brief")
graph.add_edge("format_brief", END)

# Compile into a runnable
app = graph.compile()

# Run with a sample client request
result = app.invoke({
    "client_request": "Our fintech startup needs help with market entry strategy for Southeast Asia",
    "scope_summary": "", "industry_sector": "", "engagement_brief": ""
})

print("Engagement Pipeline Result:")
print(f"  Client Request  : {result['client_request']}")
print(f"  Scope Summary   : {result['scope_summary']}")
print(f"  Industry Sector : {result['industry_sector']}")
print(f"  Brief           : {result['engagement_brief']}")

**Observe:** Each node function returns a PARTIAL state update (just the keys it changes). LangGraph merges these updates into the full state automatically. For example, `extract_scope_node` only returns `{"scope_summary": ...}` but the full state (including `client_request`, `industry_sector`, etc.) is preserved and passed to the next node.

### Try This

Change the client request to a healthcare company (e.g., "A mid-size hospital chain wants to optimize patient flow and reduce ER wait times") and re-run the cell above. Observe how `classify_industry_node` routes it to "Healthcare & Life Sciences" based on the keyword matching.

---
## Demo 2: Conditional Routing -- Routing Client Engagements by Complexity

Conditional edges let the graph take different paths based on the current state. This is how workflows adapt -- a quick benchmarking study follows a different path than a full-scale transformation program.

Here we use an LLM to classify engagement complexity and route to the appropriate workstream: **rapid assessment**, **standard engagement**, or **transformation program**.

### Step (a): Define State, LLM, and Node Functions

In [ ]:
# Demo 2a - Define state, LLM, and node functions

class EngagementRouterState(TypedDict):
    engagement_request: str
    complexity: str
    workstream_output: str

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
)

def assess_complexity_node(state: EngagementRouterState) -> dict:
    """Use LLM to classify engagement complexity."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager. Classify this client request into exactly one category: 'rapid' (2-4 week benchmarking or quick scan), 'standard' (8-12 week strategy or operations engagement), or 'transformation' (6+ month large-scale change program). Respond with just the one word."),
        HumanMessage(content=state["engagement_request"])
    ])
    complexity = response.content.strip().lower()
    print(f"  Complexity assessed: {complexity}")
    return {"complexity": complexity}

def rapid_assessment_node(state: EngagementRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey associate conducting a rapid assessment. Provide a concise 2-week diagnostic framework with key hypotheses to test."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[RAPID ASSESSMENT]\n{response.content}"}

def standard_engagement_node(state: EngagementRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager designing a standard 10-week engagement. Outline the workplan with key phases: diagnostic, analysis, recommendations, and implementation roadmap."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[STANDARD ENGAGEMENT]\n{response.content}"}

def transformation_node(state: EngagementRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner designing a large-scale transformation program. Outline the multi-phase approach including change management, capability building, and performance tracking."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[TRANSFORMATION PROGRAM]\n{response.content}"}

print("State, LLM, and 4 node functions defined.")

### Step (b): Define Routing Function and Build Graph

The routing function examines the current state and returns a string that maps to a node name. This is the decision point in the graph.

In [ ]:
# Demo 2b - Define routing function and build graph

def route_by_complexity(state: EngagementRouterState) -> str:
    """Route to the appropriate workstream based on complexity classification."""
    c = state["complexity"]
    if "rapid" in c:
        return "rapid_assessment"
    elif "transformation" in c:
        return "transformation"
    else:
        return "standard_engagement"

# Build the graph
graph = StateGraph(EngagementRouterState)
graph.add_node("assess_complexity", assess_complexity_node)
graph.add_node("rapid_assessment", rapid_assessment_node)
graph.add_node("standard_engagement", standard_engagement_node)
graph.add_node("transformation", transformation_node)

graph.set_entry_point("assess_complexity")
graph.add_conditional_edges("assess_complexity", route_by_complexity, {
    "rapid_assessment": "rapid_assessment",
    "standard_engagement": "standard_engagement",
    "transformation": "transformation"
})
graph.add_edge("rapid_assessment", END)
graph.add_edge("standard_engagement", END)
graph.add_edge("transformation", END)

app = graph.compile()
print("Graph compiled with conditional routing.")

**Key Insight:** `add_conditional_edges` takes a routing function that examines state and returns a string. That string maps to a node name via the routing dict. This is how graphs make decisions. The routing function is pure Python -- it can use any logic (LLM output, rules, thresholds) to decide the path.

### Step (c): Run Test Cases

In [ ]:
# Demo 2c - Run test cases to see routing in action

requests = [
    "We need a quick competitive benchmarking of our pricing vs. top 3 rivals",
    "Help us design a new go-to-market strategy for entering the European healthcare market",
    "We are a $20B conglomerate and need to completely restructure our operating model across 15 business units"
]

for req in requests:
    print(f"\nRequest: {req}")
    result = app.invoke({"engagement_request": req, "complexity": "", "workstream_output": ""})
    print(f"Output: {result['workstream_output'][:200]}...\n{'='*60}")

### Try This

Add a 4th complexity level called "diagnostic" for 4-6 week focused deep-dives. Modify `route_by_complexity` to check for "diagnostic" and add a corresponding node that designs a 5-week diagnostic workplan. Update the system prompt in `assess_complexity_node` to include this new category.

---
## QnA Recap: Demos 1 & 2

**Check your understanding:**

1. What happens if a node function returns a key that is not in the TypedDict? (Answer: LangGraph will raise a validation error at compile or runtime.)

2. In Demo 2, what would happen if the LLM returned "complex" instead of "rapid", "standard", or "transformation"? (Answer: The `route_by_complexity` function would fall through to the `else` clause and route to "standard_engagement" -- the default path.)

3. Why do we pass empty strings for state fields when invoking? (Answer: TypedDict requires all fields to be present in the initial state. Empty strings serve as initial values that nodes will overwrite.)

4. Could you use `add_conditional_edges` without an LLM -- just with rule-based logic? (Answer: Absolutely. The routing function is pure Python. Demo 1's `classify_industry_node` could be turned into a routing function.)

---
## Demo 3: ReAct Agent -- McKinsey Market Research Analyst

The ReAct (Reason + Act) pattern models how a research analyst works:
1. **Think** -- Reason about what data is needed
2. **Act** -- Search for intelligence using available tools
3. **Observe** -- Process the findings
4. **Repeat** until a well-supported answer emerges

Our analyst has access to simulated tools for market data, financial metrics, and competitive intelligence.

### Step (a): Define State and Simulated Tools

In [ ]:
# Demo 3a - Define state and simulated research tools

class ReActState(TypedDict):
    question: str
    thoughts: list[str]
    actions: list[str]
    observations: list[str]
    final_answer: str
    step_count: int

# Simulated McKinsey research tools (in production, these would be API calls)
def research_tool(query: str) -> str:
    """Simulated market intelligence database."""
    data = {
        "ev market size": "The global EV market was valued at $388B in 2024, projected to reach $950B by 2030 (CAGR ~16%). China leads with 60% market share.",
        "tesla competitors": "Key Tesla competitors: BYD (largest global EV seller by volume), Volkswagen Group (ID. series), Hyundai-Kia (fastest growing in Europe), and Rivian (US truck segment).",
        "ev margins": "Average EV gross margins: Tesla ~18%, BYD ~20%, legacy OEMs ~5-8% on EVs. Battery costs represent 30-40% of vehicle cost.",
        "southeast asia market": "Southeast Asia EV penetration is ~5% (2024), led by Thailand (govt subsidies) and Indonesia (nickel supply chain). Market expected to grow 25% CAGR through 2030.",
        "mckinsey frameworks": "Key McKinsey frameworks: MECE (Mutually Exclusive, Collectively Exhaustive), 7S Model, Three Horizons, Profit Pools, and Value Chain Analysis."
    }
    for key, val in data.items():
        if key in query.lower():
            return val
    return f"No data found for: {query}"

print("ReActState defined with 6 fields.")
print("Research tool loaded with 5 data entries.")
print("Available queries:", list({"ev market size", "tesla competitors", "ev margins", "southeast asia market", "mckinsey frameworks"}))

### Step (b): Define Think and Act Nodes

The think node reasons about what to research next. The act node executes the research action and records the observation.

In [ ]:
# Demo 3b - Define think and act nodes

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
)

def think_node(state: ReActState) -> dict:
    """Analyst reasons about what data to gather next."""
    context = ""
    for i in range(len(state["actions"])):
        context += f"Action: {state['actions'][i]}\nObservation: {state['observations'][i]}\n"
    
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey research analyst. Given the question and any previous research findings, decide your next step. Either search for more data (respond: ACTION: search '<query>') or provide a final synthesized answer (respond: FINAL ANSWER: <answer>). Think like a consultant -- be MECE in your approach."),
        HumanMessage(content=f"Research question: {state['question']}\n\n{context}\nWhat should I research next?")
    ])
    thought = response.content
    print(f"  Analyst thinks: {thought[:120]}...")
    return {"thoughts": state["thoughts"] + [thought]}

def act_node(state: ReActState) -> dict:
    """Execute the research action."""
    latest_thought = state["thoughts"][-1]
    
    if "FINAL ANSWER:" in latest_thought:
        answer = latest_thought.split("FINAL ANSWER:")[-1].strip()
        return {"final_answer": answer, "step_count": state["step_count"] + 1}
    
    if "ACTION: search" in latest_thought:
        query = latest_thought.split("search")[-1].strip().strip("'\"")
        observation = research_tool(query)
        print(f"  Research: search('{query}') -> {observation[:80]}...")
        return {
            "actions": state["actions"] + [f"search('{query}')"],
            "observations": state["observations"] + [observation],
            "step_count": state["step_count"] + 1
        }
    
    return {"final_answer": latest_thought, "step_count": state["step_count"] + 1}

print("think_node and act_node defined.")

### Step (c): Build the ReAct Graph and Run

In [ ]:
# Demo 3c - Build graph and run the research agent

def should_continue(state: ReActState) -> str:
    """Decide whether to keep researching or stop."""
    if state["final_answer"] or state["step_count"] >= 5:
        return "end"
    return "think"

graph = StateGraph(ReActState)
graph.add_node("think", think_node)
graph.add_node("act", act_node)

graph.set_entry_point("think")
graph.add_edge("think", "act")
graph.add_conditional_edges("act", should_continue, {"think": "think", "end": END})

app = graph.compile()

# Run the research agent
result = app.invoke({
    "question": "What is the EV market size and who are Tesla's main competitors?",
    "thoughts": [], "actions": [], "observations": [],
    "final_answer": "", "step_count": 0
})

print(f"\nAnalyst's Answer: {result['final_answer']}")
print(f"Research steps taken: {result['step_count']}")

**Gotcha:** The ReAct agent hit the 5-step limit without finding all data. In production, you would improve the search matching or add more tools. The step limit prevents infinite loops -- it is a safety valve that ensures the graph always terminates. Notice how the LLM's query phrasing (e.g., "EV market size 2023") may not match the tool's exact keys ("ev market size"). Robust tool design and better prompting both help.

---
## Demo 4: Iterative Refinement -- Refining a Recommendation Until Partner-Quality

Cycles allow a graph to loop back and improve its output. Deliverables go through multiple rounds of refinement before they reach partner-quality. This demo models that iterative process: draft a strategic recommendation, have a simulated "partner review" critique it, and loop back to refine until it meets the bar or reaches max iterations.

### Step (a): Define State and Formulate Node

In [ ]:
# Demo 4a - Define state and formulate recommendation node

class RecommendationState(TypedDict):
    strategic_question: str
    draft_recommendation: str
    partner_feedback: str
    iteration: int
    is_partner_quality: bool

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7"))
)

def formulate_recommendation(state: RecommendationState) -> dict:
    """Draft or refine a strategic recommendation."""
    if state["draft_recommendation"]:
        prompt = f"""You are a McKinsey engagement manager. Refine this strategic recommendation based on partner feedback. Make it more MECE, insight-driven, and actionable.

Current recommendation: {state['draft_recommendation']}

Partner feedback: {state['partner_feedback']}

Provide an improved recommendation (3-4 sentences, structured and concise):"""
    else:
        prompt = f"""You are a McKinsey engagement manager. Draft a strategic recommendation for this question (3-4 sentences). Be specific, actionable, and data-driven.

Strategic question: {state['strategic_question']}"""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [Iteration {state['iteration'] + 1}] Recommendation: {response.content[:100]}...")
    return {"draft_recommendation": response.content, "iteration": state["iteration"] + 1}

print("RecommendationState and formulate_recommendation defined.")

### Step (b): Define Partner Review Node

In [ ]:
# Demo 4b - Define partner review node

def partner_review(state: RecommendationState) -> dict:
    """Simulated partner review -- critiques the recommendation for quality."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner reviewing a strategic recommendation. Rate it 1-10 on: (1) MECE structure, (2) actionability, (3) insight depth. If average is 8+, respond with 'APPROVED - ready for client'. Otherwise, provide specific feedback on what to improve."),
        HumanMessage(content=state["draft_recommendation"])
    ])
    feedback = response.content
    is_approved = "APPROVED" in feedback.upper() or state["iteration"] >= 3
    print(f"  [Partner Review] {'PARTNER APPROVED' if is_approved else feedback[:80]}...")
    return {"partner_feedback": feedback, "is_partner_quality": is_approved}

print("partner_review node defined.")

### Step (c): Build the Refinement Graph and Run

In [ ]:
# Demo 4c - Build graph with cycle and run

def should_refine(state: RecommendationState) -> str:
    """If not partner-quality, loop back for another iteration."""
    if state["is_partner_quality"]:
        return "end"
    return "formulate_recommendation"

graph = StateGraph(RecommendationState)
graph.add_node("formulate_recommendation", formulate_recommendation)
graph.add_node("partner_review", partner_review)

graph.set_entry_point("formulate_recommendation")
graph.add_edge("formulate_recommendation", "partner_review")
graph.add_conditional_edges("partner_review", should_refine, {
    "formulate_recommendation": "formulate_recommendation", "end": END
})

app = graph.compile()

result = app.invoke({
    "strategic_question": "Should our client, a mid-size European bank, acquire a fintech payments startup to accelerate digital transformation?",
    "draft_recommendation": "", "partner_feedback": "", "iteration": 0, "is_partner_quality": False
})

print(f"\nFinal Recommendation (after {result['iteration']} iterations):")
print(result["draft_recommendation"])

**Observe:** The recommendation improved through iterations. Iteration 1 was generic; by iteration 3 it included specific metrics and timelines. This mirrors how partner feedback sharpens deliverables. The cycle terminates either when the partner approves (score 8+) or when the max iteration limit (3) is reached -- ensuring the graph always halts.

---
## QnA Recap: Demos 3 & 4

**Check your understanding:**

1. In Demo 3, why is `step_count` in the state rather than as a local variable? (Answer: Because state persists across the cycle. A local variable in `act_node` would reset to 0 on every call. The state is the only thing that survives between node executions.)

2. What is the difference between Demo 3's cycle (think -> act -> think) and Demo 4's cycle (formulate -> review -> formulate)? (Answer: Demo 3 is exploration-driven -- it keeps searching until it has enough information. Demo 4 is quality-driven -- it keeps refining until a quality threshold is met. Both use `add_conditional_edges` to decide when to stop.)

3. In Demo 4, why do we force approval at iteration >= 3 even if quality is not 8+? (Answer: To guarantee termination. Without a hard cap, the LLM reviewer might never approve, creating an infinite loop. Always design cycles with a maximum iteration safeguard.)

4. Could you combine ReAct (Demo 3) with iterative refinement (Demo 4)? (Answer: Yes -- you could have a research phase that gathers data, then a refinement phase that iterates on the synthesis. This is common in production consulting AI systems.)

---
## Demo 5: Human-in-the-Loop -- Partner Sign-Off Before Client Delivery

No major deliverable goes to the client without partner approval. LangGraph supports this through interrupt points where the graph pauses and waits for human approval before continuing. This demo models that gate: the system generates a client-ready action plan, pauses for partner review (simulated), and only proceeds to "deliver to client" if approved.

### Step (a): Define State and Node Functions

In [ ]:
# Demo 5a - Define state and nodes

class ClientDeliveryState(TypedDict):
    engagement_context: str
    action_plan: str
    partner_approved: bool
    delivery_output: str

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
)

def prepare_deliverable(state: ClientDeliveryState) -> dict:
    """Generate a client-ready action plan."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager. Create a structured 3-step action plan for the client. Include specific timelines, owners, and expected impact for each step."),
        HumanMessage(content=state["engagement_context"])
    ])
    print(f"  Deliverable prepared: {response.content[:120]}...")
    return {"action_plan": response.content}

def partner_signoff_node(state: ClientDeliveryState) -> dict:
    """Simulate partner review gate (auto-approve for demo)."""
    print(f"  [PARTNER REVIEW] Reviewing action plan for client delivery...")
    print(f"    Plan preview: {state['action_plan'][:200]}...")
    # In production, this would pause and wait for actual partner input
    approved = True  # Simulated approval
    print(f"  [PARTNER] {'Approved for client delivery' if approved else 'Needs revision -- send back to team'}")
    return {"partner_approved": approved}

def deliver_to_client(state: ClientDeliveryState) -> dict:
    """Package and deliver the final output to the client."""
    if not state["partner_approved"]:
        return {"delivery_output": "Delivery blocked -- partner approval required."}
    
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey partner delivering recommendations to a C-suite client. Wrap this action plan in a professional executive summary with a compelling opening and clear next steps."),
        HumanMessage(content=f"Action Plan:\n{state['action_plan']}")
    ])
    return {"delivery_output": response.content}

print("ClientDeliveryState and 3 node functions defined.")

### Step (b): Build Graph and Run

In [ ]:
# Demo 5b - Build graph and run

def route_after_review(state: ClientDeliveryState) -> str:
    return "deliver_to_client" if state["partner_approved"] else "end"

graph = StateGraph(ClientDeliveryState)
graph.add_node("prepare_deliverable", prepare_deliverable)
graph.add_node("partner_signoff", partner_signoff_node)
graph.add_node("deliver_to_client", deliver_to_client)

graph.set_entry_point("prepare_deliverable")
graph.add_edge("prepare_deliverable", "partner_signoff")
graph.add_conditional_edges("partner_signoff", route_after_review, {
    "deliver_to_client": "deliver_to_client", "end": END
})
graph.add_edge("deliver_to_client", END)

app = graph.compile()

result = app.invoke({
    "engagement_context": "A Fortune 500 CPG company needs to reduce supply chain costs by 15% within 18 months while maintaining service levels",
    "action_plan": "", "partner_approved": False, "delivery_output": ""
})

print(f"\nClient Delivery:\n{result['delivery_output'][:500]}...")

**Key Insight:** In production, `partner_signoff_node` would use LangGraph's `interrupt_before` feature to actually pause execution and wait for human input via an API. The graph state would be persisted (checkpointed), and when the partner provides approval through a web UI or Slack bot, the graph would resume from exactly where it stopped. This is what makes LangGraph suitable for long-running business processes, not just single-shot inference.

---
## QnA Recap: Demo 5

**Check your understanding:**

1. Why does `route_after_review` route to "end" (not back to prepare_deliverable) when not approved? (Answer: In this demo, rejection simply stops delivery. In a more complete system, you would route back to a revision node -- combining the human-in-the-loop pattern with the iterative refinement pattern from Demo 4.)

2. What is the difference between `interrupt_before` and our simulated gate? (Answer: `interrupt_before` actually halts graph execution and requires an external resume call with human input. Our simulation auto-approves within the same execution run -- useful for demos but not real approval workflows.)

3. In what real consulting scenarios would you need human-in-the-loop? (Answer: Partner sign-off on client deliverables, approval of financial models before sharing, compliance review of recommendations, and validation of data sources before analysis.)

---
## Summary

This demo notebook covered five core LangGraph patterns:

| # | Pattern | Key Concept | LangGraph Feature |
|---|---------|-------------|-------------------|
| 1 | Linear Pipeline | Typed state, nodes as functions, edges for sequential flow | `StateGraph`, `add_edge`, `set_entry_point` |
| 2 | Conditional Routing | Graphs that take different paths based on LLM decisions | `add_conditional_edges` |
| 3 | ReAct Agent | Reason-Act-Observe loop as a cyclic graph for tool-using agents | Cycles via conditional edges back to earlier nodes |
| 4 | Iterative Refinement | Cycles that improve output quality over multiple passes | Conditional edge that loops back or ends |
| 5 | Human-in-the-Loop | Pause points for human oversight before critical actions | `interrupt_before` (production) / gate nodes (demo) |

**Next:** In the exercises notebook, you will build these patterns yourself -- starting with a linear pipeline and progressing to a multi-stage system that combines routing, refinement, and gates.